<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"></ul></div>

In [6]:
import polars as pl
import numpy as np
from typing import Tuple, Optional


def avaliar_modelo_classificacao(
    df: pl.DataFrame,
    coluna_y: str,
    coluna_score: str,
    n_decis: int = 10
) -> pl.DataFrame:
    """
    Avalia um modelo de classificação através de análise por decil.
    
    Parameters:
    -----------
    df : pl.DataFrame
        DataFrame com os dados para avaliação
    coluna_y : str
        Nome da coluna com o target binário (0 ou 1)
    coluna_score : str
        Nome da coluna com o score/probabilidade do modelo
    n_decis : int
        Número de decis para divisão (padrão: 10)
    
    Returns:
    --------
    pl.DataFrame
        DataFrame com métricas por decil
    """
    
    # Validar inputs
    if coluna_y not in df.columns:
        raise ValueError(f"Coluna '{coluna_y}' não encontrada no DataFrame")
    if coluna_score not in df.columns:
        raise ValueError(f"Coluna '{coluna_score}' não encontrada no DataFrame")
    
    # Adicionar coluna de decil baseada no score (maior score = decil 1)
    df_work = df.select([
        pl.col(coluna_y).alias("y"),
        pl.col(coluna_score).alias("score")
    ]).with_columns([
        pl.col("score")
        .qcut(n_decis, labels=[str(i) for i in range(1, n_decis + 1)])
        .cast(pl.Int32)
        .alias("decil_temp")
    ]).with_columns([
        # Inverter a ordem dos decis (maior score = decil 1)
        (n_decis + 1 - pl.col("decil_temp")).cast(pl.Utf8).alias("decil")
    ])
    
    # Calcular métricas por decil
    metricas_decil = (
        df_work
        .group_by("decil")
        .agg([
            pl.len().alias("total"),
            pl.col("y").sum().alias("contratos"),
            (pl.len() - pl.col("y").sum()).alias("nao_contratos"),
            pl.col("y").mean().alias("taxa_conversao"),
            pl.col("score").mean().alias("score_medio")
        ])
        .with_columns([
            # Converter decil para inteiro para ordenação
            pl.col("decil").cast(pl.Int32).alias("decil_num")
        ])
        .sort("decil_num")
        .drop("decil_num")
    )
    
    # Garantir tipos consistentes
    metricas_decil = metricas_decil.with_columns([
        pl.col("total").cast(pl.Int64),
        pl.col("contratos").cast(pl.Int64),
        pl.col("nao_contratos").cast(pl.Int64)
    ])
    
    # Calcular taxa de conversão geral
    taxa_geral = float(df_work["y"].mean())
    total_contratos = int(df_work["y"].sum())
    total_nao_contratos = int(len(df_work) - total_contratos)
    
    # Adicionar Lift
    metricas_decil = metricas_decil.with_columns([
        (pl.col("taxa_conversao") / taxa_geral).alias("lift")
    ])
    
    # Calcular KS (Kolmogorov-Smirnov)
    metricas_decil = metricas_decil.with_columns([
        (pl.col("contratos").cum_sum() / total_contratos).alias("perc_acum_contratos"),
        (pl.col("nao_contratos").cum_sum() / total_nao_contratos).alias("perc_acum_nao_contratos")
    ]).with_columns([
        (pl.col("perc_acum_contratos") - pl.col("perc_acum_nao_contratos")).abs().alias("ks")
    ])
    
    # Formatar percentual de contratos
    metricas_decil = metricas_decil.with_columns([
        (pl.col("taxa_conversao") * 100).alias("perc_contratos")
    ])
    
    # Selecionar e ordenar colunas finais
    resultado = metricas_decil.select([
        "decil",
        "total",
        "contratos",
        "nao_contratos",
        pl.col("perc_contratos").round(2),
        pl.col("ks").round(4),
        pl.col("lift").round(2),
        pl.col("score_medio").round(4)
    ])
    
    return resultado


def adicionar_resumo_geral(df_metricas: pl.DataFrame, df_original: pl.DataFrame, coluna_y: str) -> pl.DataFrame:
    """
    Adiciona uma linha de resumo geral ao DataFrame de métricas.
    
    Parameters:
    -----------
    df_metricas : pl.DataFrame
        DataFrame com métricas por decil
    df_original : pl.DataFrame
        DataFrame original com todos os dados
    coluna_y : str
        Nome da coluna target
    
    Returns:
    --------
    pl.DataFrame
        DataFrame com métricas por decil + linha de resumo
    """
    
    total_geral = int(len(df_original))
    total_contratos = int(df_original[coluna_y].sum())
    total_nao_contratos = int(total_geral - total_contratos)
    taxa_geral = float((total_contratos / total_geral) * 100)
    ks_max = float(df_metricas["ks"].max())
    
    # Obter os tipos de dados do DataFrame original
    schema_original = df_metricas.schema
    
    # Criar resumo com tipos corretos desde o início
    resumo = pl.DataFrame({
        "decil": ["TOTAL"],
        "total": [int(total_geral)],
        "contratos": [int(total_contratos)],
        "nao_contratos": [int(total_nao_contratos)],
        "perc_contratos": [float(round(taxa_geral, 2))],
        "ks": [float(round(ks_max, 4))],
        "lift": [float(1.0)],
        "score_medio": [None]
    })
    
    # Garantir que os tipos de dados sejam exatamente compatíveis com o DataFrame original
    for col in resumo.columns:
        if col in schema_original:
            resumo = resumo.with_columns(pl.col(col).cast(schema_original[col]))
    
    return pl.concat([df_metricas, resumo])


def gerar_relatorio_modelo(
    df: pl.DataFrame,
    coluna_y: str,
    coluna_score: str,
    adicionar_total: bool = True
) -> Tuple[pl.DataFrame, dict]:
    """
    Gera relatório completo de avaliação do modelo.
    
    Parameters:
    -----------
    df : pl.DataFrame
        DataFrame com os dados
    coluna_y : str
        Nome da coluna target
    coluna_score : str
        Nome da coluna score
    adicionar_total : bool
        Se deve adicionar linha de total
    
    Returns:
    --------
    Tuple[pl.DataFrame, dict]
        DataFrame com métricas e dicionário com métricas globais
    """
    
    # Calcular métricas por decil
    df_metricas = avaliar_modelo_classificacao(df, coluna_y, coluna_score)
    
    # Adicionar linha de total se solicitado
    if adicionar_total:
        df_metricas = adicionar_resumo_geral(df_metricas, df, coluna_y)
    
    # Calcular métricas globais
    if adicionar_total:
        ks_max = df_metricas.filter(pl.col("decil") != "TOTAL")["ks"].max()
        decil_ks_max = df_metricas.filter((pl.col("ks") == ks_max) & (pl.col("decil") != "TOTAL"))["decil"][0]
    else:
        ks_max = df_metricas["ks"].max()
        decil_ks_max = df_metricas.filter(pl.col("ks") == ks_max)["decil"][0]
    
    metricas_globais = {
        "ks_maximo": float(ks_max),
        "decil_ks_maximo": decil_ks_max,
        "lift_top_decil": float(df_metricas.filter(pl.col("decil") == "1")["lift"][0]),
        "taxa_conversao_geral": float(df[coluna_y].mean())
    }
    
    return df_metricas, metricas_globais


# Exemplo de uso
if __name__ == "__main__":
    # Criar dados de exemplo
    np.random.seed(42)
    n_samples = 10000
    
    # Simular scores e targets com correlação
    scores = np.random.beta(2, 5, n_samples)  # Scores entre 0 e 1
    # Probabilidade de y=1 aumenta com o score
    probs = 0.1 + 0.8 * scores + np.random.normal(0, 0.1, n_samples)
    probs = np.clip(probs, 0, 1)
    y = np.random.binomial(1, probs)
    
    # Criar DataFrame Polars
    df_exemplo = pl.DataFrame({
        "target": y,
        "score_modelo": scores,
        "id": range(n_samples)
    })
    
    print("="*60)
    print("AVALIAÇÃO DO MODELO DE CLASSIFICAÇÃO")
    print("="*60)
    
    # Avaliar modelo
    df_resultado, metricas = gerar_relatorio_modelo(
        df_exemplo,
        coluna_y="target",
        coluna_score="score_modelo",
        adicionar_total=True
    )
    
    # Mostrar resultados
    print("\nAnálise por Decil:")
    print("-"*60)
    print(df_resultado)
    
    print("\nMétricas Globais:")
    print("-"*60)
    for chave, valor in metricas.items():
        if isinstance(valor, float):
            print(f"{chave}: {valor:.4f}")
        else:
            print(f"{chave}: {valor}")
    
    # Verificação de qualidade do modelo
    print("\n" + "="*60)
    print("INTERPRETAÇÃO DAS MÉTRICAS:")
    print("-"*60)
    
    ks_max = metricas["ks_maximo"]
    if ks_max > 0.40:
        print(f"KS = {ks_max:.4f} - Excelente separação")
    elif ks_max > 0.30:
        print(f"KS = {ks_max:.4f} - Boa separação")
    elif ks_max > 0.20:
        print(f"KS = {ks_max:.4f} - Separação aceitável")
    else:
        print(f"KS = {ks_max:.4f} - Separação fraca")
    
    lift_top = metricas["lift_top_decil"]
    if lift_top > 3.0:
        print(f"Lift Top Decil = {lift_top:.2f} - Excelente discriminação")
    elif lift_top > 2.0:
        print(f"Lift Top Decil = {lift_top:.2f} - Boa discriminação")
    elif lift_top > 1.5:
        print(f"Lift Top Decil = {lift_top:.2f} - Discriminação aceitável")
    else:
        print(f"Lift Top Decil = {lift_top:.2f} - Discriminação fraca")

AVALIAÇÃO DO MODELO DE CLASSIFICAÇÃO

Análise por Decil:
------------------------------------------------------------
shape: (11, 8)
┌───────┬───────┬───────────┬───────────────┬────────────────┬────────┬──────┬─────────────┐
│ decil ┆ total ┆ contratos ┆ nao_contratos ┆ perc_contratos ┆ ks     ┆ lift ┆ score_medio │
│ ---   ┆ ---   ┆ ---       ┆ ---           ┆ ---            ┆ ---    ┆ ---  ┆ ---         │
│ str   ┆ i64   ┆ i64       ┆ i64           ┆ f64            ┆ f64    ┆ f64  ┆ f64         │
╞═══════╪═══════╪═══════════╪═══════════════╪════════════════╪════════╪══════╪═════════════╡
│ 1     ┆ 1000  ┆ 605       ┆ 395           ┆ 60.5           ┆ 0.1241 ┆ 1.83 ┆ 0.5923      │
│ 2     ┆ 1000  ┆ 476       ┆ 524           ┆ 47.6           ┆ 0.1899 ┆ 1.44 ┆ 0.459       │
│ 3     ┆ 1000  ┆ 393       ┆ 607           ┆ 39.3           ┆ 0.2182 ┆ 1.19 ┆ 0.3887      │
│ 4     ┆ 1000  ┆ 345       ┆ 655           ┆ 34.5           ┆ 0.2248 ┆ 1.04 ┆ 0.3358      │
│ 5     ┆ 1000  ┆ 330       ┆ 

In [7]:
df_resultado

decil,total,contratos,nao_contratos,perc_contratos,ks,lift,score_medio
str,i64,i64,i64,f64,f64,f64,f64
"""1""",1000,605,395,60.5,0.1241,1.83,0.5923
"""2""",1000,476,524,47.6,0.1899,1.44,0.459
"""3""",1000,393,607,39.3,0.2182,1.19,0.3887
"""4""",1000,345,655,34.5,0.2248,1.04,0.3358
"""5""",1000,330,670,33.0,0.2246,1.0,0.288
…,…,…,…,…,…,…,…
"""7""",1000,264,736,26.4,0.1859,0.8,0.2035
"""8""",1000,220,780,22.0,0.136,0.67,0.1624
"""9""",1000,202,798,20.2,0.0779,0.61,0.1176
